# 5.1 Visualize Grad-CAM (Video Overlay) — BdSL Word-Level Recognition

This notebook generates a **new MP4 video** with a **Grad-CAM overlay** for your **CNN + BiLSTM + Attention** pipeline that takes **MediaPipe landmark sequences** as input.

Because your model does **not** take raw pixels, the most faithful “Grad-CAM” visualization is **temporal Grad-CAM**:
- It highlights **which frames (time steps)** contributed most to the predicted word.

**Output:**
- `output_gradcam.mp4` (same FPS as input if available)
- Overlay includes:
  - A bottom timeline heat strip (importance across 60 frames)
  - A moving indicator for the current frame
  - A border intensity proportional to Grad-CAM at that frame
  - Predicted label + confidence

> Adjust the `VIDEO_PATH`, `MODEL_PATH`, and `LABEL_PATH` as needed.


In [39]:
# --- Imports ---
import os
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mediapipe as mp


mp_holistic = mp.solutions.holistic

# --- Paths (EDIT) ---
VIDEO_PATH = "../Datasets/Processed_Data/Front/W001/W001S01F_01.mp4"                 # <-- set your input video path
MODEL_PATH = "../Models/cnn_bilstm_attention.pth" # <-- set your weight path
LABEL_PATH = "../Datasets/label.json"              # <-- set your label path
OUTPUT_PATH = "output_gradcam.mp4"

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [40]:
# --- Model definition (embedded for portability) ---
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # x: (B, T, 2*hidden_dim)
        weights = torch.softmax(self.attn(x), dim=1)  # (B, T, 1)
        return (weights * x).sum(dim=1)               # (B, 2*hidden_dim)

class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(256)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # x: (B, T, F)
        x = x.permute(0, 2, 1)      # (B, F, T)
        x = self.cnn(x)             # (B, 256, T')
        x = x.permute(0, 2, 1)      # (B, T', 256)
        x, _ = self.lstm(x)         # (B, T', 512)
        x = self.attention(x)       # (B, 512)
        return self.fc(x)


In [41]:
# --- Load labels ---
with open(LABEL_PATH, "r", encoding="utf-8") as f:
    labels = json.load(f)

NUM_CLASSES = len(labels)
INPUT_DIM = 387
MAX_FRAMES = 60

# --- Load model ---
model = CNN_BiLSTM_Attention(INPUT_DIM, NUM_CLASSES).to(device)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.eval()

print("Model loaded. Classes:", NUM_CLASSES)


Model loaded. Classes: 401


In [42]:
# --- Face landmark indices (same as your backend) ---
IMPORTANT_FACE_IDX = [
    33, 133, 159, 145, 468, 469, 263, 362, 386, 374, 471, 472,
    105, 107, 55, 65, 52, 285, 295, 282, 283, 336,
    1, 2, 98, 327, 94, 97, 168, 197,
    13, 14, 78, 308, 82, 312, 87, 317, 88, 95, 178, 191,
    80, 81, 82, 311, 310, 415, 291, 308, 324, 318, 402, 317
]


In [43]:
# --- Utility: Extract frames + landmarks ---
def extract_frames_and_landmarks(video_path, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1 or fps != fps:  # NaN check
        fps = 25.0

    frames = []
    seq = []

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        refine_face_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        while cap.isOpened() and len(frames) < max_frames:
            ret, frame_bgr = cap.read()
            if not ret:
                break

            frames.append(frame_bgr.copy())

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            results = holistic.process(frame_rgb)

            lm = []

            # Face
            if results.face_landmarks:
                for idx in IMPORTANT_FACE_IDX:
                    p = results.face_landmarks.landmark[idx]
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0.0] * len(IMPORTANT_FACE_IDX) * 3

            # Left hand
            if results.left_hand_landmarks:
                for p in results.left_hand_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0.0] * 21 * 3

            # Right hand
            if results.right_hand_landmarks:
                for p in results.right_hand_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0.0] * 21 * 3

            # Pose
            if results.pose_landmarks:
                for p in results.pose_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
            else:
                lm += [0.0] * 33 * 3

            seq.append(lm)

    cap.release()

    if len(frames) == 0:
        raise RuntimeError("No frames extracted. Check your video file.")

    # Pad seq to max_frames for your model input
    if len(seq) < max_frames:
        padding = [[0.0] * len(seq[0])] * (max_frames - len(seq))
        seq += padding

    return frames, np.array(seq, dtype=np.float32), float(fps)

def normalize_sequence(seq):
    FACE_LM, HAND_LM, POSE_LM, DIM = len(IMPORTANT_FACE_IDX), 21, 33, 3
    LEFT_HAND_START = FACE_LM * DIM
    RIGHT_HAND_START = LEFT_HAND_START + HAND_LM * DIM
    POSE_START = RIGHT_HAND_START + HAND_LM * DIM

    def get_landmark(arr, start_idx, lm_index):
        return arr[start_idx + lm_index * 3 : start_idx + lm_index * 3 + 3]

    norm_seq = []
    for frame in seq:
        pose0 = get_landmark(frame, POSE_START, 0)
        center = pose0 if np.any(pose0 != 0) else np.mean(frame.reshape(-1, 3), axis=0)
        ls = get_landmark(frame, POSE_START, 11)
        rs = get_landmark(frame, POSE_START, 12)
        scale = np.linalg.norm(ls - rs) if np.any(ls != 0) and np.any(rs != 0) else 1.0

        normalized = (frame.reshape(-1, 3) - center) / (scale + 1e-6)
        norm_seq.append(normalized.flatten())

    return np.array(norm_seq, dtype=np.float32)


In [44]:
# --- Temporal Grad-CAM for Conv1D ---
class TemporalGradCAM:
    """Grad-CAM over time for Conv1d layers."""
    def __init__(self, model, target_conv):
        self.model = model
        self.target_conv = target_conv
        self.activations = None
        self.gradients = None
        self.h1 = target_conv.register_forward_hook(self._save_activation)
        self.h2 = target_conv.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()  # (B, C, T')

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()  # (B, C, T')

    def close(self):
        self.h1.remove()
        self.h2.remove()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)  # (1, num_classes)
        probs = F.softmax(logits, dim=1)

        pred_idx = int(torch.argmax(probs, dim=1).item())
        conf = float(probs[0, pred_idx].item())

        if class_idx is None:
            class_idx = pred_idx

        score = logits[0, class_idx]
        score.backward(retain_graph=True)

        A = self.activations[0]  # (C, T')
        dA = self.gradients[0]   # (C, T')

        w = dA.mean(dim=1)  # (C,)
        cam = torch.relu((w[:, None] * A).sum(dim=0))  # (T',)

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam.detach().cpu().numpy(), pred_idx, conf

def upsample_cam(cam_tprime, target_len=MAX_FRAMES):
    cam = torch.tensor(cam_tprime)[None, None, :]  # (1,1,T')
    cam_up = F.interpolate(cam, size=target_len, mode="linear", align_corners=False)
    cam_up = cam_up[0,0].cpu().numpy()
    cam_up = np.clip(cam_up, 0, 1)
    return cam_up


In [45]:
# --- Overlay helpers ---
def build_timeline_strip(cam_vals, width, height=18):
    """
    Create a color timeline (heat strip) of shape (height, width, 3).
    cam_vals: (T,) in [0,1]
    """
    # Resize cam to match the width
    cam_1d = np.interp(np.linspace(0, len(cam_vals)-1, width), np.arange(len(cam_vals)), cam_vals)
    cam_img = (cam_1d * 255).astype(np.uint8)[None, :]  # (1, W)
    cam_img = np.repeat(cam_img, height, axis=0)        # (H, W)

    heat = cv2.applyColorMap(cam_img, cv2.COLORMAP_JET)  # (H, W, 3) BGR
    return heat

def overlay_gradcam_on_frame(frame_bgr, heat_strip, frame_idx, total_frames, intensity):
    """
    Adds:
    - bottom heat strip (full timeline)
    - a vertical indicator at current frame
    - border intensity proportional to current Grad-CAM intensity
    """
    h, w = frame_bgr.shape[:2]
    strip_h = heat_strip.shape[0]

    # 1) Put heat strip at bottom
    out = frame_bgr.copy()
    y0 = h - strip_h
    out[y0:h, 0:w] = cv2.addWeighted(out[y0:h, 0:w], 0.35, heat_strip, 0.65, 0)

    # 2) Current frame indicator line
    x = int((frame_idx / max(total_frames - 1, 1)) * (w - 1))
    cv2.line(out, (x, y0), (x, h-1), (255, 255, 255), 2)

    # 3) Border glow based on intensity
    border = int(6 + 10 * float(intensity))  # thickness
    color = (0, int(255*intensity), int(255*(1-intensity)))  # BGR
    cv2.rectangle(out, (0,0), (w-1,h-1), color, border)

    return out


In [46]:
# --- End-to-end: Generate overlay video ---
def generate_gradcam_video(video_path, output_path=OUTPUT_PATH):
    frames, raw_seq, fps = extract_frames_and_landmarks(video_path, max_frames=MAX_FRAMES)
    norm_seq = normalize_sequence(raw_seq)

    x = torch.tensor(norm_seq, dtype=torch.float32).unsqueeze(0).to(device)

    # Choose a target Conv1d layer:
    # If your model has model.cnn = nn.Sequential(...),
    # the second Conv1d is commonly at index 3: model.cnn[3]
    # If this fails, print(model) and adjust target_conv accordingly.
    target_conv = model.cnn[3]
    cam_engine = TemporalGradCAM(model, target_conv)

    try:
        cam_tprime, pred_idx, conf = cam_engine(x)
    finally:
        cam_engine.close()

    cam_60 = upsample_cam(cam_tprime, target_len=MAX_FRAMES)

    # Get label text
    label = labels[str(pred_idx)]
    title = f"Pred: {label.get('bangla','')} ({label.get('english','')}) | Conf: {conf*100:.2f}%"

    # Video writer
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    heat_strip = build_timeline_strip(cam_60, width=w, height=18)

    total = min(len(frames), MAX_FRAMES)
    for i in range(total):
        intensity = cam_60[i]
        out = overlay_gradcam_on_frame(frames[i], heat_strip, i, total, intensity)

        # Put text
        cv2.putText(out, title, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(out, f"Frame: {i+1}/{total} | CAM: {intensity:.3f}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

        writer.write(out)

    writer.release()
    print("Saved:", output_path)

# --- Run ---
generate_gradcam_video(VIDEO_PATH, OUTPUT_PATH)


Saved: output_gradcam.mp4


## Visual “Focus” Overlay on Hands/Pose (Landmark Saliency)

Temporal Grad-CAM shows **which frames** matter most.  
To show **where** the model focuses, we can visualize **input-saliency on landmarks**:

- Compute gradients of the predicted class score w.r.t. the **landmark inputs** `(T × F)`
- Convert feature-gradients to **per-landmark scores** (face/left hand/right hand/pose)
- Draw landmarks on the original frames with **color + size** based on importance
- Optionally draw **hand bounding boxes** with intensity

**Output:** `output_focus.mp4`


In [47]:
# --- Saliency on landmark inputs (T x F) ---
def landmark_saliency(model, x, class_idx=None):
    """Return saliency map over input features. x shape: (1, T, F)."""
    x = x.clone().detach().requires_grad_(True)
    logits = model(x)
    probs = F.softmax(logits, dim=1)

    pred = int(torch.argmax(probs, dim=1).item())
    conf = float(probs[0, pred].item())

    if class_idx is None:
        class_idx = pred

    score = logits[0, class_idx]
    model.zero_grad(set_to_none=True)
    score.backward()

    sal = x.grad.abs()[0]  # (T, F)
    sal = sal / (sal.max() + 1e-8)
    return sal.detach().cpu().numpy(), pred, conf

def feature_to_point_scores(frame_sal, face_n):
    """Convert a single frame saliency (F,) into per-point saliency arrays."""
    DIM = 3
    idx = 0

    face = frame_sal[idx: idx + face_n*DIM].reshape(face_n, DIM).mean(axis=1)
    idx += face_n*DIM

    lh = frame_sal[idx: idx + 21*DIM].reshape(21, DIM).mean(axis=1)
    idx += 21*DIM

    rh = frame_sal[idx: idx + 21*DIM].reshape(21, DIM).mean(axis=1)
    idx += 21*DIM

    pose = frame_sal[idx: idx + 33*DIM].reshape(33, DIM).mean(axis=1)

    return {
        "face": face,
        "left_hand": lh,
        "right_hand": rh,
        "pose": pose
    }


In [48]:
# --- Extract frames + landmarks + pixel coordinates for drawing ---
def extract_frames_landmarks_and_coords(video_path, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1 or fps != fps:
        fps = 25.0

    frames = []
    seq = []
    coords = []  # list of dicts with pixel coords

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        refine_face_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        while cap.isOpened() and len(frames) < max_frames:
            ret, frame_bgr = cap.read()
            if not ret:
                break

            h, w = frame_bgr.shape[:2]
            frames.append(frame_bgr.copy())

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            results = holistic.process(frame_rgb)

            lm = []
            c = {"face": None, "left_hand": None, "right_hand": None, "pose": None}

            # Face (selected idx)
            if results.face_landmarks:
                face_pts = []
                for idx in IMPORTANT_FACE_IDX:
                    p = results.face_landmarks.landmark[idx]
                    lm += [p.x, p.y, p.z]
                    face_pts.append((int(p.x*w), int(p.y*h)))
                c["face"] = np.array(face_pts, dtype=np.int32)
            else:
                lm += [0.0] * len(IMPORTANT_FACE_IDX) * 3

            # Left hand
            if results.left_hand_landmarks:
                pts=[]
                for p in results.left_hand_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
                    pts.append((int(p.x*w), int(p.y*h)))
                c["left_hand"] = np.array(pts, dtype=np.int32)
            else:
                lm += [0.0] * 21 * 3

            # Right hand
            if results.right_hand_landmarks:
                pts=[]
                for p in results.right_hand_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
                    pts.append((int(p.x*w), int(p.y*h)))
                c["right_hand"] = np.array(pts, dtype=np.int32)
            else:
                lm += [0.0] * 21 * 3

            # Pose
            if results.pose_landmarks:
                pts=[]
                for p in results.pose_landmarks.landmark:
                    lm += [p.x, p.y, p.z]
                    pts.append((int(p.x*w), int(p.y*h)))
                c["pose"] = np.array(pts, dtype=np.int32)
            else:
                lm += [0.0] * 33 * 3

            seq.append(lm)
            coords.append(c)

    cap.release()

    if len(frames) == 0:
        raise RuntimeError("No frames extracted. Check your video file.")

    # Pad seq and coords to max_frames
    if len(seq) < max_frames:
        pad_len = max_frames - len(seq)
        pad_vec = [0.0] * len(seq[0])
        seq += [pad_vec] * pad_len
        coords += [{"face": None, "left_hand": None, "right_hand": None, "pose": None}] * pad_len

    return frames, np.array(seq, dtype=np.float32), coords, float(fps)

def _apply_colormap_intensity(val01):
    v = int(np.clip(val01, 0, 1) * 255)
    color = cv2.applyColorMap(np.array([[v]], dtype=np.uint8), cv2.COLORMAP_JET)[0,0].tolist()
    return (int(color[0]), int(color[1]), int(color[2]))  # BGR

def draw_landmark_focus(frame_bgr, coord_dict, score_dict, alpha=0.75):
    """Draw landmark points with color/size based on saliency scores."""
    out = frame_bgr.copy()
    overlay = frame_bgr.copy()

    def draw_points(name, pts, scores, base_r):
        if pts is None or scores is None:
            return
        n = min(len(pts), len(scores))
        for i in range(n):
            s = float(scores[i])
            if s < 0.02:
                continue
            x, y = int(pts[i,0]), int(pts[i,1])
            r = int(base_r + 8*s)
            col = _apply_colormap_intensity(s)
            cv2.circle(overlay, (x,y), r, col, -1)

        # optional bbox for hands
        if name in ("left_hand","right_hand") and pts is not None and n > 0:
            xs, ys = pts[:n,0], pts[:n,1]
            x0, x1 = int(xs.min()), int(xs.max())
            y0, y1 = int(ys.min()), int(ys.max())
            pad = 10
            x0, y0 = max(x0-pad,0), max(y0-pad,0)
            x1, y1 = min(x1+pad, out.shape[1]-1), min(y1+pad, out.shape[0]-1)
            s_mean = float(np.mean(scores[:n]))
            col = _apply_colormap_intensity(s_mean)
            cv2.rectangle(overlay, (x0,y0), (x1,y1), col, 3)

    draw_points("pose", coord_dict.get("pose"), score_dict.get("pose"), base_r=3)
    draw_points("left_hand", coord_dict.get("left_hand"), score_dict.get("left_hand"), base_r=4)
    draw_points("right_hand", coord_dict.get("right_hand"), score_dict.get("right_hand"), base_r=4)
    # Face is many points; keep subtle
    draw_points("face", coord_dict.get("face"), score_dict.get("face"), base_r=2)

    out = cv2.addWeighted(overlay, alpha, out, 1-alpha, 0)
    return out


In [49]:
# --- Generate "where it focuses" video ---
FOCUS_OUTPUT_PATH = "output_focus.mp4"

def generate_focus_video(video_path, output_path=FOCUS_OUTPUT_PATH):
    frames, raw_seq, coords, fps = extract_frames_landmarks_and_coords(video_path, max_frames=MAX_FRAMES)
    norm_seq = normalize_sequence(raw_seq)

    x = torch.tensor(norm_seq, dtype=torch.float32).unsqueeze(0).to(device)

    # Saliency for predicted class
    sal_TF, pred_idx, conf = landmark_saliency(model, x)

    # Label text
    label = labels[str(pred_idx)]
    title = f"Pred: {label.get('bangla','')} ({label.get('english','')}) | Conf: {conf*100:.2f}%"

    # Writer
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    face_n = len(IMPORTANT_FACE_IDX)
    total = min(len(frames), MAX_FRAMES)

    for i in range(total):
        frame_scores = feature_to_point_scores(sal_TF[i], face_n=face_n)
        out = draw_landmark_focus(frames[i], coords[i], frame_scores, alpha=0.70)

        cv2.putText(out, title, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(out, f"Frame: {i+1}/{total}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

        writer.write(out)

    writer.release()
    print("Saved focus video:", output_path)

# --- Run to create focus video ---
generate_focus_video(VIDEO_PATH, FOCUS_OUTPUT_PATH)


Saved focus video: output_focus.mp4


## Notes / Troubleshooting

- If you get an error on `target_conv = model.cnn[3]`, your `CNN_BiLSTM_Attention` may be structured differently.
  - Print the model: `print(model)`
  - Choose the last `nn.Conv1d` layer as target.
- If your output video is blank or has wrong colors:
  - Ensure frames are BGR (OpenCV default).
- If MediaPipe is slow:
  - Reduce `MAX_FRAMES` (e.g., 30) or use shorter videos.

You can now cite this visualization in your thesis as **Temporal Grad-CAM** for sequence models.
